# Lesson 1
## Embeddings
A way to represent sentences as vectors in an embedding space(vector space).
Semantically similar vectors get placed near to eachother. 

Example:

These two sentences:
* “The dog is running”
* “A puppy is sprinting”

look different as text, but semantically they’re similar.
An embedding model maps both into nearby points in vector space. The cliser teh vectors teh similar the meaning they share.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "The dog is running in the park",
    "A puppy is sprinting outside",
    "SQL is used for databases"
]

embeddings = model.encode(sentences)

query = model.encode(['A dog is running'])

scores = cosine_similarity(query, embeddings)
print(scores)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

2026-05-15 18:07:51.363041: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778868471.548389    1833 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778868471.605948    1833 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778868472.056323    1833 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778868472.056367    1833 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778868472.056369    1833 computation_placer.cc:177] computation placer alr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

[[ 0.66139054  0.69718087 -0.0452537 ]]


The 2nd embedded sentence shows the highest cosine_similarity score which means its most semantically similar to the query. 

In [2]:
query.shape #the model represented the sentence as a 384 dimension vector 

(1, 384)

In [3]:
# Test 2: this one shows how keyword search and embeddings search differ

sentences = [
    "The server crashed because memory was exhausted",
    "The RAM overflowed causing the machine to fail",
    "The word memory appears here but unrelated"
]

embeddings = model.encode(sentences)

query = model.encode(["system failed due to low memory"])

scores = cosine_similarity(query, embeddings)

for sentence, score in zip(sentences, scores[0]):
    print(sentence, '->', score)

The server crashed because memory was exhausted -> 0.6742977
The RAM overflowed causing the machine to fail -> 0.6393724
The word memory appears here but unrelated -> 0.39389783


Here the cosine_similarity ran linearly comparing the query embeddings to each of the sentence embeddings but,

* 10 chunks → easy
* 10,000 chunks → slower
* 1,000,000 chunks → needs indexing tricks

Thats why vector databases exist. It avoids brute-force scans by using approximate nearest-neighbor methods.

# Lesson 2
## Vector Databases and Chunking
**They store:**
* vectors
* metadata
* document IDs
* chunk text

**They handle:**
* fast nearest-neighbor lookup
* filtering
* persistence

### Chunking
Diving bigger text paragraphs into smaller size chunks.

In [4]:
def chunk_text(text, chunk_size=500):

    chunks=[]
    start=0

    while start < len(text):
        end   = start + chunk_size
        chunk = text[start : end]
        chunks.append(chunk)
        start += end

    return chunks

In [5]:
text = '''Remember how an LLM works; it’s a prediction engine. The model takes sequential text as
an input and then predicts what the following token should be, based on the data it was
trained on. The LLM is operationalized to do this over and over again, adding the previously
predicted token to the end of the sequential text for predicting the following token. The next
token prediction is based on the relationship between what’s in the previous tokens and what
the LLM has seen during its training.
When you write a prompt, you are attempting to set up the LLM to predict the right sequence
of tokens. Prompt engineering is the process of designing high-quality prompts that guide
LLMs to produce accurate outputs. This process involves tinkering to find the best prompt,
optimizing prompt length, and evaluating a prompt’s writing style and structure in relation
to the task. In the context of natural language processing and LLMs, a prompt is an input
provided to the model to generate a response or prediction.
LLMs are tuned to follow instructions and are trained on large amounts of data so they can
understand a prompt and generate an answer. But LLMs aren’t perfect; the clearer your
prompt text, the better it is for the LLM to predict the next likely text. Additionally, specific
techniques that take advantage of how LLMs are trained and how LLMs work will help you get
the relevant results from LLMs
Now that we understand what prompt engineering is and what it takes, let’s dive into some
examples of the most important prompting techniques.
General prompting / zero shot
A zero-shot5
prompt is the simplest type of prompt. It only provides a description of a task
and some text for the LLM to get started with. This input could be anything: a question, a
start of a story, or instructions. The name zero-shot stands for ’no examples’.'''

In [6]:
import numpy as np

chunks = chunk_text(text=text)

embeddings = model.encode(chunks)

query = model.encode(['working of a llm'])

scores = cosine_similarity(query, embeddings)[0]

top_indices = np.argsort(scores)[::-1][:2] #top 2 best matches

for i in top_indices:
    print(chunks[i])
    print(scores[i])

Remember how an LLM works; it’s a prediction engine. The model takes sequential text as
an input and then predicts what the following token should be, based on the data it was
trained on. The LLM is operationalized to do this over and over again, adding the previously
predicted token to the end of the sequential text for predicting the following token. The next
token prediction is based on the relationship between what’s in the previous tokens and what
the LLM has seen during its training.
When 
0.5209081
you write a prompt, you are attempting to set up the LLM to predict the right sequence
of tokens. Prompt engineering is the process of designing high-quality prompts that guide
LLMs to produce accurate outputs. This process involves tinkering to find the best prompt,
optimizing prompt length, and evaluating a prompt’s writing style and structure in relation
to the task. In the context of natural language processing and LLMs, a prompt is an input
provided to the model to generate a res

lets introduce chunking with overlap by modifing out chunk_text method, for that we'll have to add a overlap attribute and a little tweak to the code should do it

In [7]:
def chunk_text_with_overlap(text, chunk_size=500, overlap=10):

    chunks=[]
    start=0

    while start < len(text):
        end   = start + chunk_size
        chunk = text[start : end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [8]:
import numpy as np

chunks = chunk_text_with_overlap(text=text)

embeddings = model.encode(chunks)

query = model.encode(['working of a llm'])

scores = cosine_similarity(query, embeddings)[0]

top_indices = np.argsort(scores)[::-1][:2] 

for i in top_indices:
    print(chunks[i])
    print(scores[i])

Remember how an LLM works; it’s a prediction engine. The model takes sequential text as
an input and then predicts what the following token should be, based on the data it was
trained on. The LLM is operationalized to do this over and over again, adding the previously
predicted token to the end of the sequential text for predicting the following token. The next
token prediction is based on the relationship between what’s in the previous tokens and what
the LLM has seen during its training.
When 
0.5209081
ate a response or prediction.
LLMs are tuned to follow instructions and are trained on large amounts of data so they can
understand a prompt and generate an answer. But LLMs aren’t perfect; the clearer your
prompt text, the better it is for the LLM to predict the next likely text. Additionally, specific
techniques that take advantage of how LLMs are trained and how LLMs work will help you get
the relevant results from LLMs
Now that we understand what prompt engineering is and what it 

Overlapping preserves continuity. Also a thing to notice is that teh score of the 2nd chunk increased which means overlapping helps preserve the continuity and flow over all the chunks.

# Lesson 3
## Real Document QA using LlamaIndex

In [9]:
# pip install llama-index

In [10]:
# pip install llama-index-embeddings-huggingface

In [11]:
# pip install chromadb

In [12]:
# !pip install pypdf

In [13]:
# !pip install llama-index-readers-file

In [14]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.readers.file import PDFReader
from pathlib import Path

dataset = Path("/kaggle/input/datasets/adwaittagalpallewar/kaggle-whitepaper-pdfs")

loader = PDFReader()

documents = []

for pdf in dataset.glob("*.pdf"):
    doc = loader.load_data(file=pdf)
    documents.extend(doc)
    
print(documents[0].text[:500])

Agents
Authors: Julia Wiesinger, Patrick Marlow  
and Vladimir Vuskovic



In [15]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

index = VectorStoreIndex.from_documents(documents)

retriever = index.as_retriever(similarity_top_k=2)

nodes = retriever.retrieve("What is prompt engineering?")

for i, node in enumerate(nodes):
    print(f"\n--- Chunk {i+1} ---\n")
    print(node.text[:1000])


--- Chunk 1 ---

Foundational Large Language Models & Text Generation
52
February 2025
Using large language models
Prompt engineering and sampling techniques have a strong influence on the performance of 
LLMs. Prompt engineering is the process of designing and refining the text inputs (prompts) 
that you feed into an LLM to achieve desired and relevant outputs. Sampling techniques 
determine the way in which output tokens are chosen and influence the correctness, 
creativity and diversity of the resulting output. We next discuss different variants of prompt 
engineering and sampling techniques as well as touch on some important parameters that 
can have a significant impact on LLM performance.
Prompt engineering 
LLMs are very powerful, but they still need guidance to unleash their full potential. Prompt 
engineering is a critical component in guiding an LLM to yield desired outputs. This might 
include grounding the model to yield factual responses or unleashing the creativity of th

# Lesson 4
## Permanent storage (ChromaDB)

In [16]:
# !pip uninstall -y chromadb opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp protobuf
# !pip install chromadb==0.5.5 \
#     opentelemetry-api==1.27.0 \
#     opentelemetry-sdk==1.27.0 \
#     opentelemetry-exporter-otlp==1.27.0 \
#     protobuf==4.25.3 -q

In [17]:
# !pip install -U chromadb==0.5.23 llama-index-vector-stores-chroma==0.5.5 -q

In [19]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

db = chromadb.PersistentClient(path='./chroma_db')
collection = db.get_or_create_collection('whitepaper')

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


**Meaning**
* creates folder chroma_db
* collection/db name = whitepapers
* saved permanently

In [20]:
#connect to llama_index

vector_store = ChromaVectorStore(chroma_collection=collection)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

now llama index writes into chroma

In [21]:
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context
)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


In [24]:
db = chromadb.PersistentClient(path="./chroma_db")

collection = db.get_collection("whitepaper")
print(collection.count())

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


260
